# 06 Planner (dry-run only)

Generate a conservative move/keep review plan from the latest review outputs.

This notebook stays **dry-run only**. It proposes target paths where the evidence is strong and keeps ambiguous rows in a structured review queue.

Policy basis for `v2_4`:
- Company root is `{COMPANY_FOLDER}` with `CORPORATE`, `TEMPLATES`, `ARCHIVE`, and `ASSETS` beneath it.
- Active assets live under `{COMPANY_FOLDER}/ASSETS/{ASSET_FOLDER}` and archived assets under `{COMPANY_FOLDER}/ARCHIVE/{YEAR}/{ASSET_FOLDER}`.
- Filenames use `{TYPEID}_{PHASE}_{DOCTYPE}_{DESCRIPTION}_{DATE}_{VERSION}_{STATUS}.{EXT}` and path limits stay conservative for Windows.
- Exact-hash duplicates are marked for review, and `SUPERSEDED` may be kept in place or moved to `_SUPERSEDED`; this notebook therefore does **not** auto-route duplicates or superseded files unless you explicitly opt in to a local convention.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'
POLICY_PATH = PROJECT_ROOT / 'policy' / 'SCH_fileserver_policy_v2_4.yaml'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUTS_DIR  =', OUTPUTS_DIR)
print('POLICY_PATH  =', POLICY_PATH)


PROJECT_ROOT = c:\00_Developement\sch-file-organizer
OUTPUTS_DIR  = c:\00_Developement\sch-file-organizer\data\outputs
POLICY_PATH  = c:\00_Developement\sch-file-organizer\policy\SCH_fileserver_policy_v2_4.yaml


In [2]:
from src.policy_loader import PolicyLoader
from src.reporting import detect_latest_outputs, load_optional_parquet, build_review_frame
from src.planner import PlanConfig, build_plan, save_plan_outputs, plan_summary

policy = PolicyLoader.from_file(POLICY_PATH)
paths = detect_latest_outputs(OUTPUTS_DIR)

print(paths)

inv = load_optional_parquet(paths.inventory_path)
cls = load_optional_parquet(paths.classification_path)
txt = load_optional_parquet(paths.text_path)

review = build_review_frame(inventory_df=inv, classification_df=cls, text_df=txt)
print('Review rows:', len(review))

display(review[['relative_path', 'rule_status', 'text_status', 'rule_reason']].head(20))


ReviewPaths(inventory_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/inventory_smoke_test.parquet'), classification_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/rule_classification_smoke_test.parquet'), text_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/inventory_with_text_smoke_test.parquet'))
Review rows: 4


,relative_path,rule_status,text_status,rule_reason
0,docs/Thumbs.db,archive_or_delete_candidate,unsupported,junk filename from archive_rules
1,docs/a.txt,move_to_special_folder,ok,duplicate hash non-canonical copy
2,docs/PV15p473-01_PM_PER_environmental-approval...,review,error,needs classification or rename mapping
3,docs/b.txt,review,ok,needs classification or rename mapping


## Planner settings

Fill in `COMPANY_FOLDER` and `ASSET_FOLDER` when you are ready to produce full target paths.

Until then, the planner will still generate asset-relative targets and keep unresolved rows in review.

Leave `ENABLE_SUPERSEDED_FOLDER` and `ENABLE_DUPLICATE_FOLDER` as `False` unless you explicitly want to adopt a local folder convention beyond what `v2_4` defines.

In [3]:
COMPANY_FOLDER = None      # e.g. 'SCH-...-123456789'
ASSET_FOLDER = None        # e.g. 'PVS15p473-01_PROJECT_LOCATION'
ROOT_MODE = 'ASSETS'       # 'ASSETS' or 'ARCHIVE'
ARCHIVE_YEAR = None        # integer only if ROOT_MODE == 'ARCHIVE'

ENABLE_SUPERSEDED_FOLDER = False
ENABLE_DUPLICATE_FOLDER = False
ENABLE_DEPRECATED_FOLDER = False

config = PlanConfig(
    company_folder=COMPANY_FOLDER,
    asset_folder=ASSET_FOLDER,
    root_mode=ROOT_MODE,
    archive_year=ARCHIVE_YEAR,
    enable_superseded_folder=ENABLE_SUPERSEDED_FOLDER,
    enable_duplicate_folder=ENABLE_DUPLICATE_FOLDER,
    enable_deprecated_folder=ENABLE_DEPRECATED_FOLDER,
)

config


PlanConfig(company_folder=None, asset_folder=None, root_mode='ASSETS', archive_year=None, enable_superseded_folder=False, enable_duplicate_folder=False, enable_deprecated_folder=False, max_ready_confidence=0.9)

In [4]:
plan = build_plan(review, policy_loader=policy, config=config)
summary = plan_summary(plan)
summary


{'rows': 4,
 'ready_rows': 0,
 'needs_user_input': 4,
 'would_change_path': 0,
 'manual_review': 2}

In [5]:
display(
    plan[[
        'relative_path',
        'rule_status',
        'planner_action',
        'planner_reason',
        'planner_confidence',
        'planner_target_relative_path',
        'planner_target_full_path',
    ]].head(25)
)

display(plan['planner_action'].value_counts(dropna=False).rename_axis('planner_action').reset_index(name='count'))


,relative_path,rule_status,planner_action,planner_reason,planner_confidence,planner_target_relative_path,planner_target_full_path
0,docs/Thumbs.db,archive_or_delete_candidate,archive_or_delete_review,junk filename from archive_rules; v2_4 says ar...,0.99,None,None
1,docs/a.txt,move_to_special_folder,review_special_folder_policy,duplicate hash non-canonical copy; v2_4 marks ...,0.98,None,None
2,docs/PV15p473-01_PM_PER_environmental-approval...,review,manual_review,needs classification or rename mapping,0.50,None,None
3,docs/b.txt,review,manual_review,needs classification or rename mapping,0.50,None,None


,planner_action,count
0,manual_review,2
1,archive_or_delete_review,1
2,review_special_folder_policy,1


In [6]:
ready = plan[plan['planner_ready']].copy()
needs_input = plan[plan['planner_needs_user_input']].copy()
changes = plan[plan['planner_would_change_path']].copy()

print('Ready rows:', len(ready))
print('Needs input:', len(needs_input))
print('Would change path:', len(changes))

display(ready[['relative_path', 'planner_action', 'planner_target_relative_path']].head(20))
display(needs_input[['relative_path', 'planner_action', 'planner_reason']].head(20))
display(changes[['relative_path', 'planner_target_relative_path']].head(20))


Ready rows: 0
Needs input: 4
Would change path: 0


,relative_path,planner_action,planner_target_relative_path


,relative_path,planner_action,planner_reason
0,docs/Thumbs.db,archive_or_delete_review,junk filename from archive_rules; v2_4 says ar...
1,docs/a.txt,review_special_folder_policy,duplicate hash non-canonical copy; v2_4 marks ...
2,docs/PV15p473-01_PM_PER_environmental-approval...,manual_review,needs classification or rename mapping
3,docs/b.txt,manual_review,needs classification or rename mapping


,relative_path,planner_target_relative_path


In [7]:
STAMP = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUTS_DIR / f'plan_dry_run_{STAMP}'
csv_path, parquet_path = save_plan_outputs(plan, output_base)
print('Saved:')
print(' -', csv_path)
print(' -', parquet_path)


Saved:
 - c:\00_Developement\sch-file-organizer\data\outputs\plan_dry_run_20260307_094350.csv
 - c:\00_Developement\sch-file-organizer\data\outputs\plan_dry_run_20260307_094350.parquet


## Read this before the next step

Good signs:
- `move_to_policy_folder` rows with sensible lifecycle targets
- `keep_in_place` rows where compliant files already sit correctly
- `manual_review` and `review_special_folder_policy` rows concentrated in legacy/messy areas

What stays intentionally unresolved here:
- rename proposals for non-compliant filenames
- archive/delete execution
- duplicate/superseded routing unless you explicitly enable that convention

Once this notebook looks good, the next layer is an **execution manifest** notebook that still does not modify files, but prepares a rollback-ready CSV of actions.
